# B-04. 로지스틱 회귀 임베딩 내보내기 (D·E용) + 미등록 판별 미리보기

B 고전 모델 중 1등(로지스틱 회귀 L2, 특성 AC, macro-F1 0.404)을 D·E가 쓸 수 있게 147+21개 split으로 내보낸다.

특성 변형 4가지를 먼저 21-fold로 비교한다.

| 변형 | 내용 | 노리는 것 |
|---|---|---|
| AC | 지금 1등 설정 | 기준 |
| AC-rms | RMS 6개 제외 | 가속도 RMS에 섞인 중력(센서 기울기) 제거. B-03에서 채널 평균을 뺐을 때 좋아진 것과 같은 이유 |
| logAC | A·C에 로그 | D-02 원인(거리가 진동 세기를 따라감) 완화: 배율 차이 → 일정한 거리 차이 |
| logAC-rms | 둘 다 | |

- 임베딩 = 학습 fold 기준으로 표준화한 특성 (로지스틱 회귀가 실제로 보는 공간), logits = 재질별 점수
- 파일: `emb_logreg_{빠진재질}_{fold}.npy`, `logits_logreg_…`, `classes_logreg_…json` (MLP와 같은 폴더·형식)
- 마지막에 D의 `ood.run_heldout`으로 MLP와 미리 비교한다 (공식 결과 기록은 D)

런타임 CPU, 전체 25~35분

In [ ]:
# 1. 레포 받기 + 데이터 + 드라이브
import os
if not os.path.exists('/content/ML-imu'):
    !git clone -q https://github.com/ekgP525/ML-imu.git /content/ML-imu
%cd /content/ML-imu
!git pull -q
!pip install -q -r requirements.txt
!python data/download.py
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/ML-imu_embeddings'   # MLP 임베딩과 같은 폴더
os.makedirs(OUT, exist_ok=True); print(OUT, len(os.listdir(OUT)), 'files')

In [ ]:
# 2. 불러오기
import sys, time, json, warnings
sys.path += ['src', 'notebooks/B_classic', 'notebooks/D_ood']
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from recordings import load_dataset
from features import extract_features
from evaluate import evaluate
from splits import load_folds, load_material_folds
from b_classic import LogRegEmbed, select_features, run_loro, export_embeddings

X, y, rec = load_dataset('data')
F_all = extract_features(X, 'ABC')
print(X.shape, F_all.shape)

In [ ]:
# 3. 특성 변형 4가지 21-fold 비교 (각 1~2분)
SPECS = ['AC', 'AC-rms', 'logAC', 'logAC-rms']
rows = []
for spec in SPECS:
    t = time.time()
    p = run_loro(LogRegEmbed, select_features(F_all, spec), y, rec)
    m = evaluate(y, p)
    rows.append({'features': spec, 'n_features': select_features(F_all, spec).shape[1], **m})
    print(f"{spec:10s} macro-F1 {m['macro_f1']:.4f}  acc {m['accuracy']:.4f}  mode {m['mode_macro_f1']:.4f}  ({time.time()-t:.0f}s)")
cmp = pd.DataFrame(rows); cmp

In [ ]:
# 4. 내보낼 변형 고르기: macro-F1 1등 (바꾸고 싶으면 BEST를 직접 적기)
BEST = cmp.sort_values('macro_f1', ascending=False).features.iloc[0]
print('내보낼 특성:', BEST)
print('한계: 21-fold 시험 점수로 골랐으므로, 고른 변형의 점수는 약간 낙관적일 수 있음')

In [ ]:
# 5. 임베딩 내보내기: 재질 하나 빼기 147 + 8종 21 (Colab 끊기면 다시 실행 → 이어서 저장)
F_best = select_features(F_all, BEST)
with open(f'{OUT}/logreg_spec.txt', 'w') as f:
    f.write(f'LogisticRegression L2 C=1 balanced, features={BEST}, embedding=standardized features ({F_best.shape[1]} dims)\n')
export_embeddings(F_best, y, rec, load_material_folds(), OUT, make_model=LogRegEmbed, name='logreg')
export_embeddings(F_best, y, rec, load_folds(), OUT, make_model=LogRegEmbed, name='logreg')
n = len([f for f in os.listdir(OUT) if f.startswith('emb_logreg_')])
print(n, '개 (기대값 168)', np.load(f'{OUT}/emb_logreg_carpet_0.npy').shape, np.load(f'{OUT}/logits_logreg_carpet_0.npy').shape)

In [ ]:
# 6. 미등록 판별 미리보기: 로지스틱 vs MLP (D의 코드 그대로, 147 split, 5~10분)
from ood import run_heldout, summarize
def loader(name):
    return lambda h, f: (np.load(f'{OUT}/emb_{name}_{h}_{f}.npy'), np.load(f'{OUT}/logits_{name}_{h}_{f}.npy'))
res = {}
for name in ['logreg', 'mlp']:
    if os.path.exists(f'{OUT}/emb_{name}_carpet_0.npy'):
        t = time.time(); res[name] = run_heldout(loader(name), y, rec, name); print(name, f'{time.time()-t:.0f}s')
ood_all = pd.concat(res.values())
print('평균 AUROC (0.5 = 찍기, 높을수록 좋음)')
display(ood_all.pivot_table(index='method', columns='model', values='auroc').round(3))
print('난이도별 평균 AUROC')
display(ood_all.pivot_table(index=['method', 'difficulty'], columns='model', values='auroc').round(3))
ood_all.to_csv(f'{OUT}/B_ood_preview_logreg_vs_mlp.csv', index=False)

In [ ]:
# 7. 빠진 재질별 AUROC (proto·energy)
ood_all[ood_all.method.isin(['proto', 'energy'])].pivot_table(
    index='heldout', columns=['method', 'model'], values='auroc').round(3)

In [ ]:
# 8. results.csv에 넣을 줄 (출력 전체를 복사해서 Claude에게)
from datetime import date
for r in rows:
    print(f"{date.today()},B,logreg_l2,8class_{r['features']},all,{r['macro_f1']:.4f},{r['accuracy']:.4f},{r['mode_macro_f1']:.4f},,,B-04 특성 변형")